<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/KNN_Gi_Cold_Spot_Berechnungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dieses Notebook berechnet eine KNN-Matrix sowie Gi*-Klassifizierung in Cold Spots, Hot Spots, Neutral Spots für
- **jede Stadt über 50.000 EW im Gesamtzeitraum 2019-2024**
    - dafür wird zuerst ein gemeinsamer gdf erstellt, der für jeden Pixel den Durchschnitts-LST-Wert aller Sommerszenen von 2019-2024 speichert (in drive: averaged_lst_per_pixel_allyears.geojson)
    - daraus werden dann lokale gdf pro Stadt erstellt und auf Basis dessen für jede Stadt eine KNN-Matrix (drive Ordner: Cold Spots Bayern > knn_weights_averaged, jede Stadt einzeln gespeichert)
    - dann wird die **Gi*-Analyse** für jede Stadt durchgeführt (München fehlt bisher noch, zu groß für Laufzeit)
    - die Gi* Ergebnisse werden klassifiziert in Cold Spots, Hot Spots, Nicht Signifikant > gespeichert im drive ordner: cold_spots_p005


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install geemap
!pip install geopandas
!pip install geopy
!pip install folium
!pip install matplotlib
!pip install numpy
!pip install pandas
!pip install rasterio
!pip install seaborn
!pip install shapely
!pip install sklearn
!pip install pysal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 35.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 4.8 MB/s eta 0:00:00
 

In [2]:
!pip install pysal

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.1/248.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━

In [3]:
# Importieren Sie notwendige Bibliotheken
import ee
import geemap
import folium
from datetime import datetime
import geopandas as gpd
from shapely.geometry import mapping, Point
import pandas as pd
import matplotlib.pyplot as plt
import pysal.lib as ps
import pysal.explore as pe
from esda.getisord import G_Local
from libpysal.weights import KNN
from sklearn.preprocessing import StandardScaler
import numpy as np
import os

/usr/local/lib/python3.11/dist-packages/spaghetti/network.py:41: FutureWarning: The next major release of pysal/spaghetti (2.0.0) will drop support for all ``libpysal.cg`` geometries. This change is a first step in refactoring ``spaghetti`` that is expected to result in dramatically reduced runtimes for network instantiation and operations. Users currently requiring network and point pattern input as ``libpysal.cg`` geometries should prepare for this simply by converting to ``shapely`` geometries.
  warnings.warn(dep_msg, FutureWarning, stacklevel=1)


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from libpysal.weights import KNN
from esda.getisord import G_Local
import numpy as np


# KNN für Mittelwert aller Sommer

**KNN für Durchschnitt der Jahre 2019-2025** erstellen => erstmal gdfs Mittelwert der Jahre, dann KNN

In [ ]:
all_cities_summer_avg_lst_per_pixel = gpd.read_file("/content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson")
display(all_cities_summer_avg_lst_per_pixel.head())

In [ ]:
# 1. Gruppierung und Mittelwertbildung
# Gruppieren nach 'city' und 'geometry' und berechnen des Mittelwerts für 'avg_summer_LST_Celsius'
# Setzen Sie 'geometry' als Index, um die Gruppierung zu vereinfachen und dann zurückzusetzen
averaged_lst_per_pixel_per_city = all_cities_summer_avg_lst_per_pixel.set_index('geometry').groupby(['city', 'geometry'])['avg_summer_LST_Celsius'].mean().reset_index()

# Konvertieren Sie das Ergebnis zurück in ein GeoDataFrame
# Annahme: Das ursprüngliche CRS des GeoDataFrames ist bekannt oder kann von der ersten Zeile abgeleitet werden.
# Wenn das ursprüngliche CRS nicht bekannt ist, müssen Sie es hier explizit festlegen.
# Beispiel: original_crs = all_cities_summer_avg_lst_per_pixel.crs
# Wenn das CRS None ist und Sie wissen, dass es sich um WGS84 (EPSG:4326) handelt:
# original_crs = "EPSG:4326"

# Verwenden Sie das CRS des ursprünglichen GeoDataFrames
original_crs = all_cities_summer_avg_lst_per_pixel.crs

averaged_lst_per_pixel_per_city = gpd.GeoDataFrame(
    averaged_lst_per_pixel_per_city,
    geometry='geometry',
    crs=original_crs # Setzen Sie das CRS des ursprünglichen GeoDataFrames
)

print("Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.")
display(averaged_lst_per_pixel_per_city.head())

Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.


,city,geometry,avg_summer_LST_Celsius
0,Aschaffenburg,POINT (9.23257 49.93527),25.827932
1,Aschaffenburg,POINT (9.2355 49.935),25.828958
2,Aschaffenburg,POINT (9.2355 49.93527),25.846390
3,Aschaffenburg,POINT (9.23591 49.935),25.713771
4,Aschaffenburg,POINT (9.23591 49.93527),25.751711


## Geodataframe 2019-2024 Durchschnitt speichern

In [ ]:
# Definieren Sie den Pfad zum Speichern der Datei in Google Drive
# Passen Sie den Ordner und Dateinamen bei Bedarf an
output_path_averaged_gdf = "/content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson"

try:
    # Speichern Sie den GeoDataFrame als GeoJSON-Datei
    averaged_lst_per_pixel_per_city.to_file(output_path_averaged_gdf, driver='GeoJSON')
    print(f"GeoDataFrame mit gemittelten LST-Werten erfolgreich gespeichert unter: {output_path_averaged_gdf}")
except Exception as e:
    print(f"Fehler beim Speichern des GeoDataFrames: {e}")

GeoDataFrame mit gemittelten LST-Werten erfolgreich gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson


gespeicherten gdf laden und damit weiterarbeiten: enthält die Mittelwerte aller verfügbaren Sommerszenen 2019-2024

In [4]:
# Definieren Sie den Pfad, von dem die Datei geladen werden soll
# Stellen Sie sicher, dass dies mit dem Pfad übereinstimmt, unter dem Sie die Datei gespeichert haben
input_path_averaged_gdf = "/content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson"

try:
    # Laden Sie den GeoDataFrame aus der GeoJSON-Datei
    loaded_averaged_lst_gdf = gpd.read_file(input_path_averaged_gdf)
    print(f"GeoDataFrame erfolgreich geladen von: {input_path_averaged_gdf}")
    print(f"Anzahl der Einträge im geladenen GeoDataFrame: {len(loaded_averaged_lst_gdf)}")
    display(loaded_averaged_lst_gdf.head())

except Exception as e:
    print(f"Fehler beim Laden des GeoDataFrames von {input_path_averaged_gdf}: {e}")
    print("Bitte überprüfen Sie den Dateipfad und stellen Sie sicher, dass die Datei existiert.")

# Nun können Sie mit 'loaded_averaged_lst_gdf' weiterarbeiten
# Zum Beispiel können Sie es in city_averaged_gdfs aufteilen oder direkt verwenden
# loaded_averaged_lst_gdf sollte die gleiche Struktur wie 'averaged_lst_per_pixel_per_city' haben

GeoDataFrame erfolgreich geladen von: /content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson
Anzahl der Einträge im geladenen GeoDataFrame: 1992069


,city,avg_summer_LST_Celsius,geometry
0,Aschaffenburg,25.827932,POINT (9.23257 49.93527)
1,Aschaffenburg,25.828958,POINT (9.2355 49.935)
2,Aschaffenburg,25.846390,POINT (9.2355 49.93527)
3,Aschaffenburg,25.713771,POINT (9.23591 49.935)
4,Aschaffenburg,25.751711,POINT (9.23591 49.93527)


In [5]:
# 2. Erstellung stadtspezifischer GeoDataFrames
# Dictionary zum Speichern der stadtspezifischen GeoDataFrames mit gemittelten Werten
city_averaged_gdfs = {}

unique_cities_averaged = loaded_averaged_lst_gdf['city'].unique()
print(f"\nEindeutige Städte im gemittelten GeoDataFrame: {list(unique_cities_averaged)}")


print("\nErstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...")

for city in unique_cities_averaged:
    print(f"  Erstelle GeoDataFrame für Stadt: {city}...")
    gdf_city_averaged = loaded_averaged_lst_gdf[
        loaded_averaged_lst_gdf['city'] == city
    ].copy() # Wichtig: Kopie erstellen

    if not gdf_city_averaged.empty:
        city_averaged_gdfs[city] = gdf_city_averaged
        print(f"    GeoDataFrame für {city} erstellt mit {len(gdf_city_averaged)} Pixeln.")
    else:
        print(f"    Keine gemittelten Daten für Stadt {city} gefunden.")

print("\nStadtspezifische GeoDataFrames mit gemittelten LST-Werten erstellt.")

# Jetzt haben Sie ein Dictionary 'city_averaged_gdfs', das für jede Stadt einen GeoDataFrame
# mit den gemittelten LST-Werten pro Pixel über alle Jahre enthält.
# Beispiel: city_averaged_gdfs['Munich'] gibt den GeoDataFrame für München mit gemittelten LST-Werten zurück.


Eindeutige Städte im gemittelten GeoDataFrame: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

Erstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...
  Erstelle GeoDataFrame für Stadt: Aschaffenburg...
    GeoDataFrame für Aschaffenburg erstellt mit 69391 Pixeln.
  Erstelle GeoDataFrame für Stadt: Augsburg...
    GeoDataFrame für Augsburg erstellt mit 164425 Pixeln.
  Erstelle GeoDataFrame für Stadt: Bamberg...
    GeoDataFrame für Bamberg erstellt mit 121604 Pixeln.
  Erstelle GeoDataFrame für Stadt: Bayreuth...
    GeoDataFrame für Bayreuth erstellt mit 149925 Pixeln.
  Erstelle GeoDataFrame für Stadt: Erlangen...
    GeoDataFrame für Erlangen erstellt mit 80705 Pixeln.
  Erstelle GeoDataFrame für Stadt: Fürth...
    GeoDataFrame für Fürth erstellt mit 60360 Pixeln.
  Erstelle GeoDataFrame für Stadt: Ingolstadt

In [ ]:
# 3. KNN-Matrix Berechnung pro Stadt
# Dictionary zum Speichern der KNN-Gewichtsmatrizen für jede Stadt
city_knn_weights_averaged = {}

# Definieren Sie die Anzahl der Nachbarn für KNN
k_neighbors = 12 # Sie können diesen Wert anpassen

print(f"\nBerechne KNN-Gewichtsmatrizen (k={k_neighbors}) für jede Stadt basierend auf gemittelten LST-Werten...")

# Schleife über die stadtspezifischen GeoDataFrames
for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"\n✨ Verarbeite Stadt: {city}...")

    # Stellen Sie sicher, dass genügend Punkte für die KNN-Analyse vorhanden sind
    min_points_for_knn = k_neighbors + 1
    if not gdf_city_averaged.empty and len(gdf_city_averaged) >= min_points_for_knn:
        # Extrahieren Sie die Koordinaten
        # Konvertiere in ein metrisches CRS vor der KNN-Berechnung, wenn es noch nicht projiziert ist
        gdf_city_spatial = gdf_city_averaged.copy() # Kopie erstellen
        if gdf_city_spatial.crs is None or gdf_city_spatial.crs.is_geographic:
             # Für Bayern ist EPSG:25832 (ETRS89 / UTM zone 32N) oft passend.
             try:
                 print(f"    Konvertiere Daten für {city} nach EPSG:25832 (UTM 32N) für metrische Distanzen.")
                 gdf_city_spatial = gdf_city_spatial.to_crs(epsg=25832)
             except Exception as crs_e:
                 print(f"    Fehler bei der CRS-Konvertierung für {city}: {crs_e}. Versuche es ohne Konvertierung, Ergebnisse könnten ungenau sein.")
                 pass # Geht weiter mit den ursprünglichen Koordinaten

        coords = np.array(list(zip(gdf_city_spatial.geometry.x, gdf_city_spatial.geometry.y)))

        # Berechne die KNN-Matrix
        try:
            w_city_averaged = KNN.from_array(coords, k=k_neighbors)
            w_city_averaged.transform = 'R' # Zeilenstandardisierung anwenden

            # Prüfen auf Konnektivität
            if w_city_averaged.n_components > 1:
                 print(f"    Warnung: Gewichtsmatrix für {city} ist nicht vollständig verbunden ({w_city_averaged.n_components} Komponenten).")

            # Speichere die Gewichtsmatrix im Dictionary
            city_knn_weights_averaged[city] = w_city_averaged
            print(f"    KNN-Matrix für {city} erfolgreich berechnet.")

        except Exception as knn_e:
            print(f"    Fehler bei der KNN-Berechnung für {city}: {knn_e}. Überspringe diese Stadt.")


    else:
        print(f"  Nicht genügend Daten ({len(gdf_city_averaged)} Punkte) für KNN-Berechnung für {city}. Benötigt mindestens {min_points_for_knn} Punkte. Überspringe.")


print("\nKNN-Berechnung für alle Städte (basierend auf gemittelten LST-Werten) abgeschlossen.")

# Das Dictionary 'city_knn_weights_averaged' enthält nun die KNN-Matrizen
# für jede Stadt, basierend auf den über die Jahre gemittelten LST-Werten pro Pixel.
# Beispiel: city_knn_weights_averaged['Munich'] gibt die KNN-Matrix für München zurück.


Berechne KNN-Gewichtsmatrizen (k=12) für jede Stadt basierend auf gemittelten LST-Werten...

✨ Verarbeite Stadt: Aschaffenburg...
    Konvertiere Daten für Aschaffenburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Aschaffenburg erfolgreich berechnet.

✨ Verarbeite Stadt: Augsburg...
    Konvertiere Daten für Augsburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Augsburg erfolgreich berechnet.

✨ Verarbeite Stadt: Bamberg...
    Konvertiere Daten für Bamberg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Bamberg erfolgreich berechnet.

✨ Verarbeite Stadt: Bayreuth...
    Konvertiere Daten für Bayreuth nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Bayreuth ist nicht vollständig verbunden (2 Komponenten).
    KNN-Matrix für Bayreuth erfolgreich berechnet.

✨ Verarbeite Stadt: Erlangen...
    Konvertiere Daten für Erlangen nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Erlangen ist nicht vollständig verbunden (10 Komponenten).
    KNN-Matrix für Erlangen erfolgreich berechnet.

✨ Verarbeite Stadt: Fürth...
    Konvertiere Daten für Fürth nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Fürth erfolgreich berechnet.

✨ Verarbeite Stadt: Ingolstadt...
    Konvertiere Daten für Ingolstadt nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Ingolstadt erfolgreich berechnet.

✨ Verarbeite Stadt: Kempten (Allgäu)...
    Konvertiere Daten für Kempten (Allgäu) nach EPSG:25832 (UTM 32N) für metrische Distanzen.


...

    Warnung: Gewichtsmatrix für Kempten (Allgäu) ist nicht vollständig verbunden (3 Komponenten).
    KNN-Matrix für Kempten (Allgäu) erfolgreich berechnet.

✨ Verarbeite Stadt: Landshut...
    Konvertiere Daten für Landshut nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Landshut erfolgreich berechnet.

✨ Verarbeite Stadt: Munich...
    Konvertiere Daten für Munich nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Munich ist nicht vollständig verbunden (3 Komponenten).
    KNN-Matrix für Munich erfolgreich berechnet.

✨ Verarbeite Stadt: Nuremberg...
    Konvertiere Daten für Nuremberg nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Nuremberg ist nicht vollständig verbunden (2 Komponenten).
    KNN-Matrix für Nuremberg erfolgreich berechnet.

✨ Verarbeite Stadt: Passau...
    Konvertiere Daten für Passau nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Passau erfolgreich berechnet.

✨ Verarbeite Stadt: Regensburg...
    Konvertiere Daten für Regensburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Regensburg erfolgreich berechnet.

✨ Verarbeite Stadt: Rosenheim...
    Konvertiere Daten für Rosenheim nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Rosenheim erfolgreich berechnet.

✨ Verarbeite Stadt: Schweinfurt...
    Konvertiere Daten für Schweinfurt nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Schweinfurt erfolgreich berechnet.

✨ Verarbeite Stadt: Würzburg...
    Konvertiere Daten für Würzburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Würzburg erfolgreich berech

### KNN-Gewichtsmatrizen speichern

Dieser Code speichert jede KNN-Gewichtsmatrix aus dem `city_knn_weights_averaged` Dictionary als separate Datei.

In [ ]:
import os
import libpysal as ps # libpysal wird zum Speichern benötigt

# Definieren Sie den Ordner, in dem die Gewichtsmatrizen gespeichert werden sollen
# Passen Sie diesen Pfad bei Bedarf an, idealerweise in Ihrem Google Drive
output_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/"
os.makedirs(output_weights_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

print(f"Speichere KNN-Gewichtsmatrizen im Ordner: {output_weights_folder}")

# Schleife über das Dictionary und speichern Sie jede Gewichtsmatrix
for city, w_matrix in city_knn_weights_averaged.items():
    if w_matrix is not None:
        # Erstellen Sie einen Dateinamen basierend auf dem Stadtnamen
        # Verwenden Sie ein Format, das von libpysal unterstützt wird, z.B. .gal oder .gwt
        # Für KNN ist .gwt (General Weights format) oft passend, aber .gal funktioniert auch oft.
        # Wir verwenden hier .gal als Beispiel.
        file_name = f"{city.replace(' ', '_')}_knn_averaged.gal"
        file_path = os.path.join(output_weights_folder, file_name)

        try:
            # Speichern Sie die Gewichtsmatrix mit libpysal.io.open()
            # Verwenden Sie nicht den context manager (with), da GalIO ihn nicht unterstützt
            f = ps.io.open(file_path, mode='w')
            f.write(w_matrix)
            f.close() # Datei explizit schließen

            print(f"  Gewichtsmatrix für {city} gespeichert unter: {file_path}")
        except Exception as e:
            print(f"  Fehler beim Speichern der Gewichtsmatrix für {city}: {e}")
    else:
        print(f"  Keine Gewichtsmatrix für {city} verfügbar zum Speichern.")

print("\nSpeichern der KNN-Gewichtsmatrizen abgeschlossen.")

Speichere KNN-Gewichtsmatrizen im Ordner: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/
  Gewichtsmatrix für Aschaffenburg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Aschaffenburg_knn_averaged.gal
  Gewichtsmatrix für Augsburg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Augsburg_knn_averaged.gal
  Gewichtsmatrix für Bamberg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Bamberg_knn_averaged.gal
  Gewichtsmatrix für Bayreuth gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Bayreuth_knn_averaged.gal
  Gewichtsmatrix für Erlangen gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Erlangen_knn_averaged.gal
  Gewichtsmatrix für Fürth gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Fürth_knn_averaged.gal
  Gewichtsmatrix für Ingolstadt gespeichert unter: /content/drive/MyDri

### KNN-Gewichtsmatrizen laden

Dieser Code lädt die zuvor gespeicherten KNN-Gewichtsmatrizen wieder in ein Dictionary.

In [ ]:
import os
import libpysal as ps # libpysal wird zum Laden benötigt

# Definieren Sie den Ordner, aus dem die Gewichtsmatrizen geladen werden sollen
input_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/"

# Dictionary zum Speichern der geladenen Gewichtsmatrizen
loaded_city_knn_weights_averaged = {}

print(f"Lade KNN-Gewichtsmatrizen aus dem Ordner: {input_weights_folder}")

# Holen Sie sich eine Liste aller Gewichtsmatrizen-Dateien im Ordner
weight_files = [f for f in os.listdir(input_weights_folder) if f.endswith(".gal")] # Passen Sie die Dateiendung an, falls Sie ein anderes Format verwendet haben

# Schleife über die Dateien und laden Sie jede Gewichtsmatrix
for file_name in weight_files:
    file_path = os.path.join(input_weights_folder, file_name)
    # Extrahiere den Stadtnamen aus dem Dateinamen
    city_name = file_name.replace("_knn_averaged.gal", "").replace("_", " ") # Passen Sie dies an Ihren Dateinamen an

    try:
        # Laden Sie die Gewichtsmatrix
        w_matrix = ps.io.open(file_path, mode='r').read()
        # Wenn die Matrix zeilenstandardisiert war beim Speichern, bleibt sie das beim Laden.
        # Stellen Sie sicher, dass die Transformation korrekt ist, falls notwendig.
        # w_matrix.transform = 'R' # Optional: Zeilenstandardisierung erneut anwenden, falls nicht erhalten

        # Speichere die geladene Gewichtsmatrix im Dictionary
        loaded_city_knn_weights_averaged[city_name] = w_matrix
        print(f"  Gewichtsmatrix für {city_name} erfolgreich geladen.")
    except Exception as e:
        print(f"  Fehler beim Laden der Gewichtsmatrix aus {file_path}: {e}")

print("\nLaden der KNN-Gewichtsmatrizen abgeschlossen.")

# Das Dictionary 'loaded_city_knn_weights_averaged' enthält nun die geladenen Gewichtsmatrizen.
# Sie können dieses Dictionary anstelle von 'city_knn_weights_averaged' in Ihrer Gi* Analyse verwenden.

Lade KNN-Gewichtsmatrizen aus dem Ordner: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/
  Gewichtsmatrix für Fürth erfolgreich geladen.
  Gewichtsmatrix für Bamberg erfolgreich geladen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Kempten (Allgäu) erfolgreich geladen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Erlangen erfolgreich geladen.
  Gewichtsmatrix für Aschaffenburg erfolgreich geladen.
  Gewichtsmatrix für Ingolstadt erfolgreich geladen.
  Gewichtsmatrix für Landshut erfolgreich geladen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Bayreuth erfolgreich geladen.
  Gewichtsmatrix für Augsburg erfolgreich geladen.
  Gewichtsmatrix für Munich erfolgreich geladen.
  Gewichtsmatrix für Nuremberg erfolgreich geladen.
  Gewichtsmatrix für Passau erfolgreich geladen.
  Gewichtsmatrix für Rosenheim erfolgreich geladen.
  Gewichtsmatrix für Schweinfurt erfolgreich geladen.
  Gewichtsmatrix für Würzburg erfolgreich geladen.
  Gewichtsmatrix für Regensburg erfolgreich geladen.

Laden der KNN-Gewichtsmatrizen abgeschlossen.


4. **Ergebnisse speichern/anzeigen**:

Die gemittelten GeoDataFrames pro Stadt sind im Dictionary `city_averaged_gdfs` gespeichert.
Die KNN-Gewichtsmatrizen pro Stadt sind im Dictionary `city_knn_weights_averaged` gespeichert.

Sie können diese Ergebnisse nun speichern oder weiter analysieren (z.B. Gi* Statistik berechnen).


# Gi* Analyse

In [ ]:
from esda.getisord import G_Local

In [ ]:
print("\n📋 Städte in city_averaged_gdfs:")
for c in sorted(city_averaged_gdfs.keys()):
    print(f"- {c}")

print("\n📋 Städte in loaded_city_knn_weights_averaged:")
for c in sorted(loaded_city_knn_weights_averaged.keys()):
    print(f"- {c}")



📋 Städte in city_averaged_gdfs:
- Aschaffenburg
- Augsburg
- Bamberg
- Bayreuth
- Erlangen
- Fürth
- Ingolstadt
- Kempten (Allgäu)
- Landshut
- Munich
- Nuremberg
- Passau
- Regensburg
- Rosenheim
- Schweinfurt
- Würzburg

📋 Städte in loaded_city_knn_weights_averaged:
- Aschaffenburg
- Augsburg
- Bamberg
- Bayreuth
- Erlangen
- Fürth
- Ingolstadt
- Kempten (Allgäu)
- Landshut
- Munich
- Nuremberg
- Passau
- Regensburg
- Rosenheim
- Schweinfurt
- Würzburg


In [ ]:
output_dir = "gi_star_results"
os.makedirs(output_dir, exist_ok=True)

def calculate_hotspots(gdf, w):
    gdf = gdf.copy()
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs
    return gdf

cities_gdfs = set(city_averaged_gdfs.keys())
cities_weights = set(loaded_city_knn_weights_averaged.keys())

print("\n📋 Städte in city_averaged_gdfs:", sorted(cities_gdfs))
print("📋 Städte in loaded_city_knn_weights_averaged:", sorted(cities_weights))

for city in sorted(cities_gdfs):
    if city not in loaded_city_knn_weights_averaged:
        print(f"⚠️  Kein Gewicht für {city} gefunden — überspringe.")
        continue

    print(f"\n✨ Bearbeite {city}...")

    gdf = city_averaged_gdfs[city]
    w = loaded_city_knn_weights_averaged[city]

    # Gi* berechnen
    result_gdf = calculate_hotspots(gdf, w)

    out_path = os.path.join(output_dir, f"{city.replace(' ', '_')}.geojson")

    result_gdf.to_file(out_path, driver="GeoJSON")
    print(f"✅ Ergebnis für {city} gespeichert: {out_path}")

    # Speicher freigeben
    del result_gdf



📋 Städte in city_averaged_gdfs: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']
📋 Städte in loaded_city_knn_weights_averaged: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

✨ Bearbeite Aschaffenburg...
✅ Ergebnis für Aschaffenburg gespeichert: gi_star_results/Aschaffenburg.geojson

✨ Bearbeite Augsburg...
✅ Ergebnis für Augsburg gespeichert: gi_star_results/Augsburg.geojson

✨ Bearbeite Bamberg...
✅ Ergebnis für Bamberg gespeichert: gi_star_results/Bamberg.geojson

✨ Bearbeite Bayreuth...


start city jetzt immer ab da wo vorherige Sitzung abgestürzt ist

In [ ]:
output_dir = "gi_star_results"
os.makedirs(output_dir, exist_ok=True)

def calculate_hotspots(gdf, w):
    gdf = gdf.copy()
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs
    return gdf

cities_gdfs = sorted(city_averaged_gdfs.keys())
cities_weights = set(loaded_city_knn_weights_averaged.keys())

print("\n📋 Städte in city_averaged_gdfs:", cities_gdfs)
print("📋 Städte in loaded_city_knn_weights_averaged:", sorted(cities_weights))

# nur Städte >= "Bayreuth" (alphabetisch) nehmen
start_city = "Passau"
start_index = cities_gdfs.index(start_city)
cities_to_process = cities_gdfs[start_index:]

print(f"\n🚀 Starte Berechnung ab: {start_city}")

for city in cities_to_process:
    if city not in loaded_city_knn_weights_averaged:
        print(f"⚠️  Kein Gewicht für {city} gefunden — überspringe.")
        continue

    print(f"\n✨ Bearbeite {city}...")

    gdf = city_averaged_gdfs[city]
    w = loaded_city_knn_weights_averaged[city]

    # Gi* berechnen
    result_gdf = calculate_hotspots(gdf, w)

    out_path = os.path.join(output_dir, f"{city.replace(' ', '_')}.geojson")

    result_gdf.to_file(out_path, driver="GeoJSON")
    print(f"✅ Ergebnis für {city} gespeichert: {out_path}")

    # Speicher freigeben
    del result_gdf



📋 Städte in city_averaged_gdfs: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']
📋 Städte in loaded_city_knn_weights_averaged: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

🚀 Starte Berechnung ab: Passau

✨ Bearbeite Passau...
✅ Ergebnis für Passau gespeichert: gi_star_results/Passau.geojson

✨ Bearbeite Regensburg...
✅ Ergebnis für Regensburg gespeichert: gi_star_results/Regensburg.geojson

✨ Bearbeite Rosenheim...
✅ Ergebnis für Rosenheim gespeichert: gi_star_results/Rosenheim.geojson

✨ Bearbeite Schweinfurt...
✅ Ergebnis für Schweinfurt gespeichert: gi_star_results/Schweinfurt.geojson
⚠️  Kein Gewicht für Würzburg gefunden — überspringe.


## Workaround für einzelne Städte
(mit Umlaut, und Munich, Nuremberg (Größe)

Fürth

In [ ]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Kopie, um das Original nicht zu ändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Hol dir die Daten für Fürth
gdf_fuerth = city_averaged_gdfs["Fürth"]

# Find the correct key for Fürth in loaded_city_knn_weights_averaged
w_fuerth = None
for key in loaded_city_knn_weights_averaged.keys():
    if "Fürth" in key or "Fürth" in key: # Check for both standard and potentially different umlaut representation
        w_fuerth = loaded_city_knn_weights_averaged[key]
        print(f"Found matching key for Fürth: {key}")
        break

if w_fuerth is not None:
    # Berechne Hotspots
    gdf_fuerth_result = calculate_hotspots(gdf_fuerth, w_fuerth)

    # Define the output path for the Fürth results GeoJSON file in your Google Drive
    output_folder = "/content/drive/MyDrive/Cold Spots Bayern/gi_results/"
    os.makedirs(output_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

    file_name = "Fürth_gi_results.geojson"
    output_path_fuerth = os.path.join(output_folder, file_name)

    try:
        # Save the GeoDataFrame as a GeoJSON file
        gdf_fuerth_result.to_file(output_path_fuerth, driver='GeoJSON')
        print(f"Gi* results for Fürth successfully saved to: {output_path_fuerth}")
    except Exception as e:
        print(f"Error saving Gi* results for Fürth: {e}")

    # Ergebnisse anschauen (optional)
    # print(gdf_fuerth_result.head())
else:
    print("Could not find the weights matrix for Fürth in loaded_city_knn_weights_averaged.")

Found matching key for Fürth: Fürth
Gi* results for Fürth successfully saved to: /content/drive/MyDrive/Cold Spots Bayern/gi_results/Fürth_gi_results.geojson


Kempten

In [ ]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Kopie, um das Original nicht zu ändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Hol dir die Daten für Kempten (Allgäu)
gdf_kempten = city_averaged_gdfs["Kempten (Allgäu)"]

# Find the correct key for Kempten (Allgäu) in loaded_city_knn_weights_averaged
w_kempten = None
for key in loaded_city_knn_weights_averaged.keys():
    if "Kempten (Allgäu)" in key or "Kempten (Allgäu)" in key: # Check for both standard and potentially different umlaut representation
        w_kempten = loaded_city_knn_weights_averaged[key]
        print(f"Found matching key for Kempten (Allgäu): {key}")
        break

if w_kempten is not None:
    # Berechne Hotspots
    gdf_kempten_result = calculate_hotspots(gdf_kempten, w_kempten)

    # Define the output path for the Kempten (Allgäu) results GeoJSON file in your Google Drive
    output_folder = "/content/drive/MyDrive/Cold Spots Bayern/gi_results/"
    os.makedirs(output_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

    file_name = "Kempten_Allgaeu_gi_results.geojson".replace(" ", "_").replace("(", "").replace(")", "") # Clean up filename
    output_path_kempten = os.path.join(output_folder, file_name)

    try:
        # Save the GeoDataFrame as a GeoJSON file
        gdf_kempten_result.to_file(output_path_kempten, driver='GeoJSON')
        print(f"Gi* results for Kempten (Allgäu) successfully saved to: {output_path_kempten}")
    except Exception as e:
        print(f"Error saving Gi* results for Kempten (Allgäu): {e}")

    # Ergebnisse anschauen (optional)
    # print(gdf_kempten_result.head())
else:
    print("Could not find the weights matrix for Kempten (Allgäu) in loaded_city_knn_weights_averaged.")

Found matching key for Kempten (Allgäu): Kempten (Allgäu)
Gi* results for Kempten (Allgäu) successfully saved to: /content/drive/MyDrive/Cold Spots Bayern/gi_results/Kempten_Allgaeu_gi_results.geojson


Würzburg

In [ ]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Kopie, um das Original nicht zu ändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Hol dir die Daten für Würzburg
gdf_wuerzburg = city_averaged_gdfs["Würzburg"]

# Find the correct key for Würzburg in loaded_city_knn_weights_averaged
# We can iterate through the keys and check if the city name is a substring
w_wuerzburg = None
for key in loaded_city_knn_weights_averaged.keys():
    if "Würzburg" in key or "Würzburg" in key: # Check for both standard and potentially different umlaut representation
        w_wuerzburg = loaded_city_knn_weights_averaged[key]
        print(f"Found matching key for Würzburg: {key}")
        break

if w_wuerzburg is not None:
    # Berechne Hotspots
    gdf_wuerzburg_result = calculate_hotspots(gdf_wuerzburg, w_wuerzburg)

    # Ergebnisse anschauen
    print(gdf_wuerzburg_result.head())
else:
    print("Could not find the weights matrix for Würzburg in loaded_city_knn_weights_averaged.")

München (hat noch nicht geklappt, Sitzung stürzt immer ab) weiter mit Rest außer München erstmal)

erstmal: nur KNN-Matrix für München laden, dann GI*

In [7]:
import libpysal as ps

# Definieren Sie den Pfad zur KNN-Gewichtsmatrix-Datei für München in Google Drive
munich_weights_path = "/content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Munich_knn_averaged.gal"

print(f"Lade KNN-Gewichtsmatrix für München von: {munich_weights_path}")

try:
    # Laden Sie die Gewichtsmatrix
    # Verwenden Sie nicht den context manager (with), da GalIO ihn nicht unterstützt
    w_munich_loaded = ps.io.open(munich_weights_path, mode='r').read()
    # Stellen Sie sicher, dass die Transformation korrekt ist, falls notwendig.
    # w_munich_loaded.transform = 'R' # Optional: Zeilenstandardisierung erneut anwenden, falls nicht erhalten

    print(f"  Gewichtsmatrix für München erfolgreich geladen.")
except Exception as e:
    print(f"  Fehler beim Laden der Gewichtsmatrix für München von {munich_weights_path}: {e}")
    w_munich_loaded = None # Setze w_munich_loaded auf None im Fehlerfall

# Überprüfen Sie, ob die Matrix geladen wurde, bevor Sie fortfahren
if w_munich_loaded is not None:
    # Entferne den Zugriff auf .k, da dieses Attribut nicht existiert
    print(f"Geladene Gewichtsmatrix für München: {w_munich_loaded.n} Beobachtungen.")
else:
    print("Konnte die Gewichtsmatrix für München nicht laden. Die Gi* Analyse kann nicht durchgeführt werden.")

Lade KNN-Gewichtsmatrix für München von: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Munich_knn_averaged.gal
  Gewichtsmatrix für München erfolgreich geladen.
Geladene Gewichtsmatrix für München: 349956 Beobachtungen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  w = W(neighbors, id_order=ids)


In [8]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Kopie, um das Original nicht zu ändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Hol dir die Daten für Munich
gdf_munich = city_averaged_gdfs["Munich"]

# Verwenden Sie die explizit geladene Gewichtsmatrix für München
# Stellen Sie sicher, dass 'w_munich_loaded' aus der vorherigen Zelle verfügbar und nicht None ist
if w_munich_loaded is not None:
    print("\nBerechne Hotspots für München mit der geladenen Gewichtsmatrix...")

    # Berechne Hotspots
    gdf_munich_result = calculate_hotspots(gdf_munich, w_munich_loaded)

    # Define the output path for the Munich results GeoJSON file in your Google Drive
    output_folder = "/content/drive/MyDrive/Cold Spots Bayern/gi_results/"
    os.makedirs(output_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

    file_name = "Munich_gi_results.geojson"
    output_path_munich = os.path.join(output_folder, file_name)

    try:
        # Save the GeoDataFrame as a GeoJSON file
        gdf_munich_result.to_file(output_path_munich, driver='GeoJSON')
        print(f"Gi* results for Munich successfully saved to: {output_path_munich}")
    except Exception as e:
        print(f"Error saving Gi* results for Munich: {e}")

    # Ergebnisse anschauen (optional)
    # print(gdf_munich_result.head())
else:
    print("Die Gi* Analyse für München konnte nicht durchgeführt werden, da die Gewichtsmatrix nicht geladen wurde.")


Berechne Hotspots für München mit der geladenen Gewichtsmatrix...
Gi* results for Munich successfully saved to: /content/drive/MyDrive/Cold Spots Bayern/gi_results/Munich_gi_results.geojson


In [ ]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Kopie, um das Original nicht zu ändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Hol dir die Daten für Munich
gdf_munich = city_averaged_gdfs["Munich"]
# Since Munich is in the loaded_city_knn_weights_averaged dictionary with the standard key,
# we can directly access its weight matrix.
w_munich = loaded_city_knn_weights_averaged["Munich"]

# Berechne Hotspots
gdf_munich_result = calculate_hotspots(gdf_munich, w_munich)

# Define the output path for the Munich results GeoJSON file in your Google Drive
output_folder = "/content/drive/MyDrive/Cold Spots Bayern/gi_results/"
os.makedirs(output_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

file_name = "Munich_gi_results.geojson"
output_path_munich = os.path.join(output_folder, file_name)

try:
    # Save the GeoDataFrame as a GeoJSON file
    gdf_munich_result.to_file(output_path_munich, driver='GeoJSON')
    print(f"Gi* results for Munich successfully saved to: {output_path_munich}")
except Exception as e:
    print(f"Error saving Gi* results for Munich: {e}")

# Ergebnisse anschauen (optional)
# print(gdf_munich_result.head())

## Cold und Hot Spots klassifizieren

Ergebnisse aus Ordner gi_results laden (= alle Sädte außer München)

In [ ]:
import os

# Definieren Sie den Pfad zu dem Ordner in Google Drive, der die GeoJSON-Dateien enthält
input_folder_gi_results = "/content/drive/MyDrive/Cold Spots Bayern/gi_results" # Verwenden Sie den Ordner mit den chunked Ergebnissen

# Dictionary zum Speichern der geladenen GeoDataFrames pro Stadt
loaded_gi_results_gdfs = {}

print(f"Lade Gi* Analyse-Ergebnisse aus dem Ordner: {input_folder_gi_results}")

# Holen Sie sich eine Liste aller GeoJSON-Dateien im Ordner
# Filtern Sie nach Dateien, die mit '.geojson' enden und nicht temporär sind
gi_result_files = [f for f in os.listdir(input_folder_gi_results) if f.endswith(".geojson") and not f.startswith('.')]

print(f"Gefundene GeoJSON-Dateien: {gi_result_files}")

# Schleife über die Dateien und laden Sie jede in ein separates GeoDataFrame
for file_name in gi_result_files:
    file_path = os.path.join(input_folder_gi_results, file_name)
    # Extrahiere den Stadtnamen aus dem Dateinamen (passen Sie dies ggf. an Ihr Namensschema an)
    # Annahme: Dateiname ist Stadtname_gi_results_chunked.geojson
    city_name = file_name.replace("_gi_results_chunked.geojson", "").replace("_", " ")

    try:
        print(f"  Lade Ergebnisse für {city_name} von {file_name}...")
        # Laden Sie den GeoDataFrame
        gdf_city_results = gpd.read_file(file_path)

        # Speichere den geladenen GeoDataFrame im Dictionary
        loaded_gi_results_gdfs[city_name] = gdf_city_results
        print(f"    Ergebnisse für {city_name} erfolgreich geladen ({len(gdf_city_results)} Zeilen).")
    except Exception as e:
        print(f"  Fehler beim Laden der Ergebnisse aus {file_path}: {e}")

print("\nLaden der Gi* Analyse-Ergebnisse abgeschlossen.")

# Das Dictionary 'loaded_gi_results_gdfs' enthält nun die geladenen GeoDataFrames,
# einen für jede Stadt, mit den Gi* Analyse-Ergebnissen aus der Chunking-Methode.
# Sie können auf die Daten für eine bestimmte Stadt zugreifen, z. B.:
# gdf_munich_gi = loaded_gi_results_gdfs['Munich']

# Hinweis zum Speicherverbrauch:
# Obwohl die Daten pro Stadt separat gespeichert werden, kann das Halten aller
# GeoDataFrames im Speicher (im Dictionary) bei sehr vielen großen Städten
# immer noch zu hohem RAM-Verbrauch führen. Wenn Sie mit sehr großen Datensätzen
# arbeiten und weiterhin Speicherprobleme auftreten, müssen Sie möglicherweise
# die Verarbeitung jeder Stadt einzeln durchführen und die Ergebnisse
# direkt nach der Verarbeitung speichern, ohne alle GeoDataFrames gleichzeitig
# im Speicher zu halten.

Lade Gi* Analyse-Ergebnisse aus dem Ordner: /content/drive/MyDrive/Cold Spots Bayern/gi_results
Gefundene GeoJSON-Dateien: ['Aschaffenburg_gi_results.geojson', 'Augsburg_gi_results.geojson', 'Bamberg_gi_results.geojson', 'Erlangen_gi_results.geojson', 'Bayreuth_gi_results.geojson', 'Landshut_gi_results.geojson', 'Ingolstadt_gi_results.geojson', 'Passau_gi_results.geojson', 'Rosenheim_gi_results.geojson', 'Schweinfurt_gi_results.geojson', 'Regensburg_gi_results.geojson', 'Nuremberg_gi_results.geojson', 'Würzburg_gi_results.geojson', 'Fürth_gi_results.geojson', 'Kempten_Allgaeu_gi_results.geojson']
  Lade Ergebnisse für Aschaffenburg gi results.geojson von Aschaffenburg_gi_results.geojson...
    Ergebnisse für Aschaffenburg gi results.geojson erfolgreich geladen (69391 Zeilen).
  Lade Ergebnisse für Augsburg gi results.geojson von Augsburg_gi_results.geojson...
    Ergebnisse für Augsburg gi results.geojson erfolgreich geladen (164425 Zeilen).
  Lade Ergebnisse für Bamberg gi results.g

nur die Gi* Ergebnisse von München laden

In [9]:
loaded_gi_results_munich = gpd.read_file("/content/drive/MyDrive/Cold Spots Bayern/gi_results/Munich_gi_results.geojson")

In [10]:
print(loaded_gi_results_munich.head())

     city  avg_summer_LST_Celsius       Gi*                   geometry
0  Munich               34.020015 -0.078997  POINT (11.71328 48.14557)
1  Munich               27.635723 -2.304778  POINT (11.64901 48.21754)
2  Munich               27.528056 -2.306209  POINT (11.64861 48.21753)
3  Munich               27.581130 -2.300726  POINT (11.64882 48.21752)
4  Munich               27.459600 -2.321848  POINT (11.64841 48.21753)


In [ ]:
# Find the correct key for Nuremberg in the dictionary
nuremberg_key = None
for key in loaded_gi_results_gdfs.keys():
    if "Nuremberg" in key:
        nuremberg_key = key
        break

if nuremberg_key:
    gdf_nuremberg_gi = loaded_gi_results_gdfs[nuremberg_key]
    print(gdf_nuremberg_gi.head())
else:
    print("Could not find the GeoDataFrame for Nuremberg in loaded_gi_results_gdfs.")

        city  avg_summer_LST_Celsius       Gi*                   geometry
0  Nuremberg               36.744860  0.213090  POINT (11.00053 49.54017)
1  Nuremberg               36.307240  0.183716   POINT (11.00051 49.5399)
2  Nuremberg               35.970451  0.168999   POINT (11.0001 49.53991)
3  Nuremberg               35.754261  0.155883  POINT (10.99969 49.53992)
4  Nuremberg               36.238215  0.171663   POINT (10.9997 49.54019)


## nach Cold Spots mit p<0,05 filtern

In [ ]:
# Dictionary zum Speichern der klassifizierten Gi* Ergebnisse pro Stadt
classified_gi_results_gdfs = {}

# Definieren Sie die zu verwendenden Schwellenwerte für die Signifikanz (Z-Werte)
# Gängige Schwellenwerte für z-Werte:
# ~1.65 für 90% Konfidenz (p<0.10)
# ~1.96 für 95% Konfidenz (p<0.05)
# ~2.58 für 99% Konfidenz (p<0.01)
z_score_thresholds = {
    'p010': 1.645, # ~90% Konfidenz
    'p005': 1.96,  # ~95% Konfidenz
    'p001': 2.576  # ~99% Konfidenz
}

print(f"Klassifiziere Gi* Ergebnisse für jede Stadt basierend auf Z-Werten und Schwellenwerten: {z_score_thresholds.keys()}...")

# Schleife über die geladenen Gi* Ergebnisse pro Stadt
for city, gdf_gi_results in loaded_gi_results_gdfs.items():
    print(f"\nKlassifiziere Ergebnisse für Stadt: {city}...")

    # Stellen Sie sicher, dass die erforderliche Spalte vorhanden ist
    if 'Gi*' in gdf_gi_results.columns:

        # Erstellen Sie eine Kopie, um die ursprünglichen Daten nicht zu ändern
        gdf_classified = gdf_gi_results.copy()

        # Iterieren Sie über die definierten Z-Wert-Schwellenwerte
        for label, threshold in z_score_thresholds.items():
            # Definieren Sie den Spaltennamen für die Klassifizierung bei diesem Schwellenwert
            sig_col_name = f'Gi_Star_sig_{label}'

            # Setzen Sie einen Standardwert für diese Signifikanzstufe
            gdf_classified[sig_col_name] = 'Nicht signifikant'

            # Klassifizieren Sie Hot Spots: Gi* (Z-Wert) > threshold
            gdf_classified.loc[
                gdf_classified['Gi*'] > threshold,
                sig_col_name
            ] = f'Hot Spot (z>{threshold:.3f})'

            # Klassifizieren Sie Cold Spots: Gi* (Z-Wert) < -threshold
            gdf_classified.loc[
                gdf_classified['Gi*'] < -threshold,
                sig_col_name
            ] = f'Cold Spot (z<{-threshold:.3f})'

        # Speichere den klassifizierten GeoDataFrame im Dictionary
        classified_gi_results_gdfs[city] = gdf_classified
        print(f"  Klassifizierung für {city} abgeschlossen für Z-Wert-Schwellenwerte.")

        # Neu: Zeigen Sie die Verteilung der Klassifizierungen für die Stadt für alle Schwellenwerte
        print(f"    Verteilung der Klassifizierungen für {city}:")
        for label in z_score_thresholds.keys():
             sig_col_name = f'Gi_Star_sig_{label}'
             print(f"      {label}:")
             print(gdf_classified[sig_col_name].value_counts().to_string()) # use to_string() to ensure all counts are shown


    else:
        print(f"  Überspringe Klassifizierung für {city}: Erforderliche 'Gi*' Spalte nicht gefunden.")


print("\nKlassifizierung der Gi* Ergebnisse für alle Städte abgeschlossen.")

# Das Dictionary 'classified_gi_results_gdfs' enthält nun die GeoDataFrames
# für jede Stadt mit zusätzlichen Spalten für die Klassifizierung basierend auf Z-Werten.
# Beispiel: classified_gi_results_gdfs['Munich']['Gi_Star_sig_p005'].value_counts()

Klassifiziere Gi* Ergebnisse für jede Stadt basierend auf Z-Werten und Schwellenwerten: dict_keys(['p010', 'p005', 'p001'])...

Klassifiziere Ergebnisse für Stadt: Aschaffenburg gi results.geojson...
  Klassifizierung für Aschaffenburg gi results.geojson abgeschlossen für Z-Wert-Schwellenwerte.
    Verteilung der Klassifizierungen für Aschaffenburg gi results.geojson:
      p010:
Gi_Star_sig_p010
Nicht signifikant       65933
Hot Spot (z>1.645)       3359
Cold Spot (z<-1.645)       99
      p005:
Gi_Star_sig_p005
Nicht signifikant     68042
Hot Spot (z>1.960)     1349
      p001:
Gi_Star_sig_p001
Nicht signifikant     69114
Hot Spot (z>2.576)      277

Klassifiziere Ergebnisse für Stadt: Augsburg gi results.geojson...
  Klassifizierung für Augsburg gi results.geojson abgeschlossen für Z-Wert-Schwellenwerte.
    Verteilung der Klassifizierungen für Augsburg gi results.geojson:
      p010:
Gi_Star_sig_p010
Nicht signifikant       155980
Hot Spot (z>1.645)        7682
Cold Spot (z<-1.645)

Filtern und speichern für München (p005 und p010)

In [11]:
import os

# Definieren Sie die Z-Wert-Schwellenwerte für p<0.05 und p<0.10
# Für einen zweiseitigen Test:
# p < 0.05 entspricht |Z| > 1.96
# p < 0.10 entspricht |Z| > 1.645
# Da wir an Cold Spots interessiert sind (negative Z-Werte), verwenden wir:
# p < 0.05: Z < -1.96
# p < 0.10: Z < -1.645
z_score_thresholds_munich = {
    'p010': 1.645, # ~90% Konfidenz für Cold/Hot Spots (Z < -1.645 oder Z > 1.645)
    'p005': 1.96   # ~95% Konfidenz für Cold/Hot Spots (Z < -1.960 oder Z > 1.960)
}

# Stellen Sie sicher, dass loaded_gi_results_munich verfügbar ist und die 'Gi*' Spalte enthält
if 'Gi*' in loaded_gi_results_munich.columns:
    print("Klassifiziere Gi* Ergebnisse für München basierend auf Z-Werten und Schwellenwerten...")

    # Erstellen Sie eine Kopie, um das Original nicht zu ändern
    gdf_munich_classified = loaded_gi_results_munich.copy()

    # Iterieren Sie über die definierten Z-Wert-Schwellenwerte
    for label, threshold in z_score_thresholds_munich.items():
        # Definieren Sie den Spaltennamen für die Klassifizierung bei diesem Schwellenwert
        sig_col_name = f'Gi_Star_sig_{label}'

        # Setzen Sie einen Standardwert für diese Signifikanzstufe
        gdf_munich_classified[sig_col_name] = 'Nicht signifikant'

        # Klassifizieren Sie Hot Spots: Gi* (Z-Wert) > threshold
        gdf_munich_classified.loc[
            gdf_munich_classified['Gi*'] > threshold,
            sig_col_name
        ] = f'Hot Spot (z>{threshold:.3f})'

        # Klassifizieren Sie Cold Spots: Gi* (Z-Wert) < -threshold
        gdf_munich_classified.loc[
            gdf_munich_classified['Gi*'] < -threshold,
            sig_col_name
        ] = f'Cold Spot (z<{-threshold:.3f})'

    print("  Klassifizierung für München abgeschlossen für Z-Wert-Schwellenwerte.")

    # Neu: Zeigen Sie die Verteilung der Klassifizierungen für München für alle Schwellenwerte
    print("    Verteilung der Klassifizierungen für München:")
    for label in z_score_thresholds_munich.keys():
         sig_col_name = f'Gi_Star_sig_{label}'
         print(f"      {label}:")
         print(gdf_munich_classified[sig_col_name].value_counts().to_string()) # use to_string() to ensure all counts are shown

    # --- Speichern der gefilterten Cold Spots ---

    # Ordner für p<0.05 Cold Spots
    output_folder_cold_spots_p005 = "/content/drive/MyDrive/Cold Spots Bayern/cold_spots_p005/"
    os.makedirs(output_folder_cold_spots_p005, exist_ok=True)

    # Ordner für p<0.10 Cold Spots
    output_folder_cold_spots_p010 = "/content/drive/MyDrive/Cold Spots Bayern/cold_spots_p010/"
    os.makedirs(output_folder_cold_spots_p010, exist_ok=True)

    # Filtern und Speichern für p<0.05
    cold_spots_p005_munich = gdf_munich_classified[
        gdf_munich_classified['Gi_Star_sig_p005'] == 'Cold Spot (z<-1.960)'
    ].copy()

    if not cold_spots_p005_munich.empty:
        file_name_p005 = "Munich_cold_spots_p005.geojson"
        file_path_p005 = os.path.join(output_folder_cold_spots_p005, file_name_p005)
        try:
            cold_spots_p005_munich.to_file(file_path_p005, driver='GeoJSON')
            print(f"  Cold Spots (z<-1.960) für München gespeichert unter: {file_path_p005} ({len(cold_spots_p005_munich)} Zeilen)")
        except Exception as e:
            print(f"  Fehler beim Speichern der Cold Spots (p005) für München: {e}")
    else:
        print("  Keine Cold Spots (z<-1.960) für München gefunden.")

    # Filtern und Speichern für p<0.10
    cold_spots_p010_munich = gdf_munich_classified[
        gdf_munich_classified['Gi_Star_sig_p010'] == 'Cold Spot (z<-1.645)'
    ].copy()

    if not cold_spots_p010_munich.empty:
        file_name_p010 = "Munich_cold_spots_p010.geojson"
        file_path_p010 = os.path.join(output_folder_cold_spots_p010, file_name_p010)
        try:
            cold_spots_p010_munich.to_file(file_path_p010, driver='GeoJSON')
            print(f"  Cold Spots (z<-1.645) für München gespeichert unter: {file_path_p010} ({len(cold_spots_p010_munich)} Zeilen)")
        except Exception as e:
            print(f"  Fehler beim Speichern der Cold Spots (p010) für München: {e}")
    else:
         print("  Keine Cold Spots (z<-1.645) für München gefunden.")


else:
    print("Überspringe Klassifizierung für München: Erforderliche 'Gi*' Spalte nicht gefunden in 'loaded_gi_results_munich'.")

Klassifiziere Gi* Ergebnisse für München basierend auf Z-Werten und Schwellenwerten...
  Klassifizierung für München abgeschlossen für Z-Wert-Schwellenwerte.
    Verteilung der Klassifizierungen für München:
      p010:
Gi_Star_sig_p010
Nicht signifikant       314755
Cold Spot (z<-1.645)     24975
Hot Spot (z>1.645)       10226
      p005:
Gi_Star_sig_p005
Nicht signifikant       329681
Cold Spot (z<-1.960)     14841
Hot Spot (z>1.960)        5434
  Cold Spots (z<-1.960) für München gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/cold_spots_p005/Munich_cold_spots_p005.geojson (14841 Zeilen)
  Cold Spots (z<-1.645) für München gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/cold_spots_p010/Munich_cold_spots_p010.geojson (24975 Zeilen)


## Speichern der Cold spots mit z< -1,96 (p<0,05)
in drive ordner cold_spots_p005

In [ ]:
# Definieren Sie den Ordner in Google Drive, in dem die gefilterten Cold Spot Ergebnisse gespeichert werden sollen
output_folder_cold_spots_p005 = "/content/drive/MyDrive/Cold Spots Bayern/cold_spots_p005/"
os.makedirs(output_folder_cold_spots_p005, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

# Dictionary zum Speichern der gefilterten Cold Spot GeoDataFrames pro Stadt
cold_spots_p005_gdfs = {}

print(f"Filtere nach Cold Spots (z<-1.960) und speichere Ergebnisse im Ordner: {output_folder_cold_spots_p005}")

# Schleife über die klassifizierten Gi* Ergebnisse pro Stadt
for city, gdf_classified in classified_gi_results_gdfs.items():
    print(f"\nFiltere Cold Spots für Stadt: {city}...")

    # Stellen Sie sicher, dass die Klassifizierungsspalte für p005 vorhanden ist
    sig_col_name = 'Gi_Star_sig_p005'
    if sig_col_name in gdf_classified.columns:
        # Filtern Sie die Zeilen, die als 'Cold Spot (z<-1.960)' klassifiziert wurden
        # Beachten Sie den exakten String aus der Klassifizierung in Zelle FnXwATPwsSUa
        cold_spots_gdf = gdf_classified[
            gdf_classified[sig_col_name] == 'Cold Spot (z<-1.960)'
        ].copy() # Wichtig: Kopie erstellen, um SettingWithCopyWarning zu vermeiden

        if not cold_spots_gdf.empty:
            # Speichere den gefilterten GeoDataFrame im Dictionary
            cold_spots_p005_gdfs[city] = cold_spots_gdf

            # Erstellen Sie einen Dateinamen basierend auf dem Stadtnamen
            # Ersetzen Sie Leerzeichen und Sonderzeichen für einen gültigen Dateinamen
            # Passen Sie die Ersetzung an die tatsächlichen Stadtnamen in den Keys an
            file_name = f"{city.replace(' gi results.geojson', '').replace(' ', '_').replace('(', '').replace(')', '').replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss').replace('ü', 'ue').replace('ö', 'oe').replace('ä', 'ae')}_cold_spots_p005.geojson"
            file_path = os.path.join(output_folder_cold_spots_p005, file_name)

            try:
                # Speichern Sie den gefilterten GeoDataFrame als GeoJSON-Datei
                cold_spots_gdf.to_file(file_path, driver='GeoJSON')
                print(f"  Cold Spots (z<-1.960) für {city} gespeichert unter: {file_path} ({len(cold_spots_gdf)} Zeilen)")
            except Exception as e:
                print(f"  Fehler beim Speichern der Cold Spots für {city}: {e}")
        else:
            print(f"  Keine Cold Spots (z<-1.960) für {city} gefunden.")

    else:
        print(f"  Überspringe Filtern für {city}: Klassifizierungsspalte '{sig_col_name}' nicht gefunden.")


print("\nFiltern und Speichern der Cold Spot Ergebnisse (z<-1.960) abgeschlossen.")

# Das Dictionary 'cold_spots_p005_gdfs' enthält nun die GeoDataFrames
# für jede Stadt, die nur die Cold Spots mit z<-1.960 enthalten.
# Die Ergebnisse wurden auch in Ihrem Google Drive gespeichert.

Filtere nach Cold Spots (z<-1.960) und speichere Ergebnisse im Ordner: /content/drive/MyDrive/Cold Spots Bayern/cold_spots_p005/

Filtere Cold Spots für Stadt: Aschaffenburg gi results.geojson...
  Keine Cold Spots (z<-1.960) für Aschaffenburg gi results.geojson gefunden.

Filtere Cold Spots für Stadt: Augsburg gi results.geojson...
  Cold Spots (z<-1.960) für Augsburg gi results.geojson gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/cold_spots_p005/Augsburg_cold_spots_p005.geojson (84 Zeilen)

Filtere Cold Spots für Stadt: Bamberg gi results.geojson...
  Keine Cold Spots (z<-1.960) für Bamberg gi results.geojson gefunden.

Filtere Cold Spots für Stadt: Erlangen gi results.geojson...
  Cold Spots (z<-1.960) für Erlangen gi results.geojson gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/cold_spots_p005/Erlangen_cold_spots_p005.geojson (1063 Zeilen)

Filtere Cold Spots für Stadt: Bayreuth gi results.geojson...
  Cold Spots (z<-1.960) für Bayreuth gi results.geojs

keine Cold Spots in Aschaffenburg und Bamberg, der Rest im Drive Ordner gespeichert

# Weiterverarbeitung der Cold-Spot-Ergebnisse (in neuem Notebook)

Schritte und weitere Ziele:
- Cold Spots visualisieren
- OSMnx Erreichbarkeitsanalysen
- evtl Verknüpfung mit LULC Daten (LCZs?)